# NumPy support in Numba

**Optional deep dive, about 15 minutes.**

Numba is designed to be used with NumPy and complement its capabilities.  Numba supports:

* Passing NumPy arrays as arguments, including structured dtypes
* Creating compiled ufuncs and generalized ufuncs
* Using a large subset of NumPy functions in nopython mode

In [ ]:
import numpy as np
import numba
from numba import njit

## Numba Specialization by Dtype

Numba automatically uses multiple dispatch on compiled functions to allow different specialized implementations of the same function to be used.
This means that Numba generates a different version of the function for each type of the arguments.
Suppose we have a function that clamps values to zero if they are below a particular magnitude:

In [ ]:
@njit
def zero_clamp(x, threshold):
    # this function is designed for 1D arrays
    out = np.empty_like(x)
    for i in range(out.shape[0]):
        if np.abs(x[i]) > threshold:
            out[i] = x[i]
        else:
            out[i] = 0
    return out        

In [ ]:
a_small = np.linspace(0, 1, 50)
zero_clamp(a_small, 0.3)

Now let's benchmark some different kinds of array inputs.  We'll try:

* int64
* float32
* float32 with a stride (elements not contiguous in memory)

In [ ]:
n = 10000
a_int64 = np.arange(n).astype(np.int64)
a_float32 = np.linspace(0, 1, n, dtype=np.float32)
a_float32_strided = np.linspace(0, 1, 2*n, dtype=np.float32)[::2]  # view of every other element

In [ ]:
%timeit -n 10 -r 3 zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 zero_clamp(a_float32_strided, 0.3)

We see different performance characteristics for each of these cases, even though they have the same number of input elements.  Numba generated different machine code for each situation, which we can see if we look at the `.signatures` attribute of the compiled function:

In [ ]:
zero_clamp.signatures

When printed as strings, Numba array types have the form: `array(dtype, dimensions, layout)`.  The first signature therefore corresponds to a 1D array of float64 with C style layout (row-major order, no gaps between elements).  The next two signatures are similar, but for `int64` and `float32` arrays.  The final signature indicates an "any" layout array, which usually happens when you slice an array, and it no longer has a C or FORTRAN memory layout.

We can compare to a pure NumPy implementation and see the speed improvement that Numba has achieved through a combination of specialization and elimination of temporary arrays:

In [ ]:
def np_zero_clamp(x, threshold):
    return np.where(np.abs(x) > threshold, x, 0)

In [ ]:
%timeit -n 10 -r 3 np_zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 np_zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 np_zero_clamp(a_float32_strided, 0.3)

## Creating Ufuncs

Universal functions, typically called "ufuncs" for short, are functions that broadcast an elementwise operation across input arrays of varying numbers of dimensions.  Most NumPy functions are ufuncs, and Numba makes it easy to compile custom ufuncs using the `@vectorize` decorator.

In [ ]:
from numba import vectorize

In [ ]:
@vectorize
def ufunc_zero_clamp(x, threshold):
    if np.abs(x) > threshold:
        return x
    else:
        return 0

In [ ]:
%timeit -n 10 -r 3 ufunc_zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 ufunc_zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 ufunc_zero_clamp(a_float32_strided, 0.3)

Note that for this simple ufunc, Numba is not as fast as the function with the manual looping, and in some cases, is the same speed as the example that called NumPy directly.  This is not surprising as this function is very simple, and NumPy *also uses compiled ufuncs*.  Numba `@vectorize` is generally most effective when creating ufuncs that are not a simple combination of existing NumPy operations.